In [1]:
import pandas as pd
import numpy as np
from pgmpy.models import BayesianModel
from pgmpy.models import BayesianNetwork
from pgmpy.inference import VariableElimination, ApproxInference, BeliefPropagation
from pgmpy.estimators import MaximumLikelihoodEstimator
from pgmpy.estimators import BayesianEstimator
from pgmpy.estimators import HillClimbSearch
from pgmpy.estimators import BDeuScore, K2Score, BicScore
from pgmpy.metrics import structure_score
from pgmpy.utils import get_example_model
from pgmpy.estimators import ScoreCache
from pgmpy.inference.CausalInference import CausalInference
import networkx as nx
import itertools
import math
import networkx as nx
import matplotlib.pyplot as plt

In [2]:
from same_decision_probability_calculation import *
from minimum_information_loss_partition import *
from utils import *

from monte_carlo_sdp import *

In [3]:
from pgmpy.utils import get_example_model

# Loading Models

In [4]:
import pandas as pd
from ucimlrepo import fetch_ucirepo
from pgmpy.models import NaiveBayes
from pgmpy.estimators import MaximumLikelihoodEstimator

# ── VOTING ──────────────────────────────────────────────────────────────────
voting = fetch_ucirepo(id=105)
df_voting = pd.concat([voting.data.features, voting.data.targets], axis=1)
df_voting.columns = [c.strip() for c in df_voting.columns]

# Replace '?' missing values — Naive Bayes needs complete data
df_voting = df_voting.replace('?', pd.NA).dropna()

# All values must be strings/categories for pgmpy
df_voting = df_voting.astype(str)

target_voting = 'Class'   # 'democrat' / 'republican'

voting_model = NaiveBayes()
voting_model.fit(df_voting, target_voting,
                 estimator=MaximumLikelihoodEstimator)

# ── CHESS ────────────────────────────────────────────────────────────────────
chess = fetch_ucirepo(id=22)
df_chess = pd.concat([chess.data.features, chess.data.targets], axis=1)
df_chess = df_chess.astype(str)

target_chess = 'skach' 

chess_model = NaiveBayes()
chess_model.fit(df_chess, target_chess,
                estimator=MaximumLikelihoodEstimator)

In [5]:
# cast models to pgmpy BayesianNetwork for compatibility with our code
voting_model = BayesianNetwork(voting_model.edges())
chess_model = BayesianNetwork(chess_model.edges())

# fit
voting_model.fit(df_voting, estimator=MaximumLikelihoodEstimator)
chess_model.fit(df_chess, estimator=MaximumLikelihoodEstimator)

In [6]:
# find binary variables in chess df
binary_vars_chess = [col for col in df_chess.columns if df_chess[col].nunique() == 2]
print(f"Binary variables in Chess dataset: {binary_vars_chess}")

Binary variables in Chess dataset: ['bkblk', 'bknwy', 'bkon8', 'bkona', 'bkspr', 'bkxbq', 'bkxcr', 'bkxwp', 'blxwp', 'bxqsq', 'cntxt', 'dsopp', 'dwipd', 'katri', 'mulch', 'qxmsq', 'r2ar8', 'reskd', 'reskr', 'rimmx', 'rkxwp', 'rxmsq', 'simpl', 'skach', 'skewr', 'skrxp', 'spcop', 'stlmt', 'thrsk', 'wkcti', 'wkna8', 'wknck', 'wkovl', 'wkpos', 'wtoeg']


In [7]:
child_model = get_example_model('child')
# cardinality of all nodes
cardinalities_child = {node: len(child_model.get_cpds(node).state_names[node]) for node in child_model.nodes()}
cardinalities_child

{'BirthAsphyxia': 2,
 'HypDistrib': 2,
 'HypoxiaInO2': 3,
 'CO2': 3,
 'ChestXray': 5,
 'Grunting': 2,
 'LVHreport': 2,
 'LowerBodyO2': 3,
 'RUQO2': 3,
 'CO2Report': 2,
 'XrayReport': 5,
 'Disease': 6,
 'GruntingReport': 2,
 'Age': 3,
 'LVH': 2,
 'DuctFlow': 3,
 'CardiacMixing': 4,
 'LungParench': 3,
 'LungFlow': 3,
 'Sick': 2}

In [8]:
alarm_model = get_example_model('alarm')
barley_model = get_example_model('barley')
child_model = get_example_model('child')
insurance_model = get_example_model('insurance')
hailfinder_model = get_example_model('hailfinder')
hepar_model = get_example_model('hepar2')
win95pts_model = get_example_model('win95pts')


In [9]:
# cardinality of all nodes in model
test_model = hailfinder_model
cardinalities = {node: len(test_model.get_cpds(node).state_names[node]) for node in test_model.nodes()}
cardinalities

{'N0_7muVerMo': 4,
 'SubjVertMo': 4,
 'QGVertMotion': 4,
 'CombVerMo': 4,
 'AreaMeso_ALS': 4,
 'SatContMoist': 4,
 'RaoContMoist': 4,
 'CombMoisture': 4,
 'AreaMoDryAir': 4,
 'VISCloudCov': 3,
 'IRCloudCover': 3,
 'CombClouds': 3,
 'CldShadeOth': 3,
 'AMInstabMt': 3,
 'InsInMt': 3,
 'WndHodograph': 4,
 'OutflowFrMt': 3,
 'MorningBound': 3,
 'Boundaries': 3,
 'CldShadeConv': 3,
 'CompPlFcst': 3,
 'CapChange': 3,
 'LoLevMoistAd': 4,
 'InsChange': 3,
 'MountainFcst': 3,
 'Date': 6,
 'Scenario': 11,
 'ScenRelAMCIN': 2,
 'MorningCIN': 4,
 'AMCINInScen': 3,
 'CapInScen': 3,
 'ScenRelAMIns': 6,
 'LIfr12ZDENSd': 4,
 'AMDewptCalPl': 3,
 'AMInsWliScen': 3,
 'InsSclInScen': 3,
 'ScenRel3_4': 5,
 'LatestCIN': 4,
 'LLIW': 4,
 'CurPropConv': 4,
 'ScnRelPlFcst': 11,
 'PlainsFcst': 3,
 'N34StarFcst': 3,
 'R5Fcst': 3,
 'Dewpoints': 7,
 'LowLLapse': 4,
 'MeanRH': 3,
 'MidLLapse': 3,
 'MvmtFeatures': 4,
 'RHRatio': 3,
 'SfcWndShfDis': 7,
 'SynForcng': 5,
 'TempDis': 4,
 'WindAloft': 4,
 'WindFieldMt': 2,

In [ ]:
# Very Large Nets
andes_model = get_example_model('andes')
link_model = get_example_model('link')
pathfinder_model = get_example_model('pathfinder')

In [10]:
child_model.name = 'child'
insurance_model.name = 'insurance'
alarm_model.name = 'alarm'
#hepar_model.name = 'hepar'
hailfinder_model.name = 'hailfinder'
#win95pts_model.name = 'win95pts'
#barley_model.name = 'barley'
#voting_model.name = 'voting'
#chess_model.name = 'chess'
#andes_model.name = 'andes'
#link_model.name = 'link'
#pathfinder_model.name = 'pathfinder'

In [16]:
# show binary variables in very large models
for model in [andes_model, link_model, pathfinder_model]:
    binary_vars = [node for node in model.nodes() if len(model.get_cpds(node).state_names[node]) == 2]
    print(f"Binary variables in {model.name} model: {binary_vars}")

Binary variables in andes model: ['GOAL_2', 'SNode_3', 'SNode_4', 'SNode_5', 'SNode_6', 'SNode_7', 'DISPLACEM0', 'RApp1', 'GIVEN_1', 'RApp2', 'SNode_8', 'SNode_9', 'SNode_10', 'SNode_11', 'SNode_12', 'SNode_13', 'SNode_14', 'SNode_15', 'SNode_16', 'SNode_17', 'SNode_18', 'SNode_19', 'NEED1', 'SNode_20', 'GRAV2', 'SNode_21', 'VALUE3', 'SNode_24', 'SLIDING4', 'SNode_25', 'CONSTANT5', 'SNode_26', 'KNOWN6', 'VELOCITY7', 'SNode_47', 'RApp3', 'KNOWN8', 'RApp4', 'SNode_27', 'COMPO16', 'GOAL_48', 'TRY12', 'TRY11', 'GOAL_49', 'CHOOSE19', 'GOAL_50', 'SYSTEM18', 'SNode_51', 'KINEMATI17', 'SNode_52', 'IDENTIFY10', 'GOAL_53', 'IDENTIFY9', 'SNode_28', 'TRY13', 'TRY14', 'TRY15', 'VAR20', 'SNode_29', 'SNode_31', 'GIVEN21', 'SNode_33', 'SNode_34', 'VECTOR27', 'APPLY32', 'GOAL_56', 'CHOOSE35', 'GOAL_57', 'MAXIMIZE34', 'SNode_59', 'AXIS33', 'SNode_60', 'WRITE31', 'GOAL_61', 'WRITE30', 'GOAL_62', 'RESOLVE37', 'GOAL_63', 'NEED36', 'SNode_64', 'SNode_41', 'SNode_42', 'IDENTIFY39', 'SNode_43', 'RESOLVE38', '

In [11]:
def select_optimal_target_node_old(bn):
    """
    Selects a target node deeply embedded in the network (highest degree).
    """
    best_node = None
    max_degree = -1
    
    for node in bn.nodes():
        # Ensure it's a binary node
        if len(bn.get_cpds(node).state_names[node]) != 2:
            continue
            
        degree = len(bn.get_parents(node)) + len(bn.get_children(node))
        if degree > max_degree:
            max_degree = degree
            best_node = node
            
    # Fallback if no binary nodes exist (rare)
    if best_node is None:
        return random.choice(list(bn.nodes()))
        
    return best_node

#antes_target = select_optimal_target_node_old(andes_model)
#link_target = select_optimal_target_node_old(link_model)
#pathfinder_target = select_optimal_target_node_old(pathfinder_model)
#print(f"Selected target node for ANDES: {antes_target}")
#print(f"Selected target node for LINK: {link_target}")
#print(f"Selected target node for PATHFINDER: {pathfinder_target}")

# Run Experiment

In [12]:
def get_target(model):
    targets = {
        'child': 'Sick',
        'alarm': 'HYPOVOLEMIA',
        'barley': 'pesticid',
        'insurance': 'Theft',
        'hailfinder': 'ScenRelAMCIN',
        'hepar': 'hepatomegaly',
        'win95pts': 'PrtMem',
        'voting': 'Class',
        'chess': 'skach',
        'andes': 'NEED36',
        'link': 'N21_d_m',
        'pathfinder': 'F97'
    }

    return targets[model.name]

## ensure all targets are present in the respective models
#for model in [child_model, alarm_model, barley_model, insurance_model, hailfinder_model, hepar_model, win95pts_model, andes_model, link_model, pathfinder_model]:
#    print(f"Checking target node for model '{model.name}'...")
#    target = get_target(model)
#    if target not in model.nodes():
#        raise ValueError(f"Target node '{target}' not found in model '{model.name}'")

def get_h_ratio(model):
    ratios = {
        'child': 0.5,
        'alarm': 0.30,
        'hepar': 0.20,
        'barley': 0.20,
        'mildew': 0.30,
        'water': 0.30,
        'hailfinder': 0.30,
        'win95pts': 0.20,
        'insurance': 0.40,
        'voting': 0.5,
        'chess': 0.9 #era 0.86
    }
    return ratios[model.name]
    



In [13]:
models_to_run = [child_model, hailfinder_model]

In [14]:
#models_to_run = [chess_model]

In [15]:
for model in models_to_run:
    print(f"Model '{model.name}' has {len(model.nodes())} variables.")

Model 'child' has 20 variables.
Model 'hailfinder' has 56 variables.


In [16]:
len(models_to_run)

2

In [17]:
all_targets_are_binary = True
for bn in models_to_run:
    #print(f"\n=== BN: {bn.name} ===")
    target = get_target(bn)
    if target is None:
        #print(f"--> No binary target defined for {bn.name}, skipping.")
        continue
    target_states = bn.get_cpds(target).state_names[target]
    if len(target_states) != 2:
        #print(f"--> Target '{target}' in {bn.name} is not binary (States: {target_states}), skipping.")
        all_targets_are_binary = False
        continue
    #print(f"Available states for target '{target}': {target_states}")
    target_value = target_states[1] if len(target_states) > 1 else target_states[0]
    #print(f"Target Node: {target}, Target Value: {target_value}")

print(f"\nAll targets are binary: {all_targets_are_binary}")


All targets are binary: True


In [18]:
for bn in models_to_run:
    print(f"\n=== BN: {bn.name} ===")
    all_nodes = list(bn.nodes())
    
    target = get_target(bn)
    if target is None:
        print(f"--> No binary target defined for {bn.name}, skipping.")
        continue
    target_states = bn.get_cpds(target).state_names[target]
    target_value = target_states[1] if len(target_states) > 1 else target_states[0]
    print(f"Target Node: {target}, Target Value: {target_value}")
    
    available_nodes = [n for n in all_nodes if n != target]
    print(f"H ratio: {get_h_ratio(bn)}")
    n_hidden = max(1, int(len(available_nodes) * get_h_ratio(bn)))
    print(f"using {n_hidden} H variables")


=== BN: child ===
Target Node: Sick, Target Value: no
H ratio: 0.5
using 9 H variables

=== BN: hailfinder ===
Target Node: ScenRelAMCIN, Target Value: CThruK
H ratio: 0.3
using 16 H variables


In [ ]:
import time
from xml.parsers.expat import model
import tracemalloc

models_to_run = [child_model, alarm_model,hailfinder_model]

def run_for_time(func, *args, **kwargs):
    """Runs natively at maximum speed to record pure execution time."""
    start_time = time.time()
    try:
        result = func(*args, **kwargs)
        return result, (time.time() - start_time), True
    except Exception as e:
        return None, np.nan, False # Failed

def run_for_memory(func, *args, **kwargs):
    """Runs with tracemalloc to record peak memory. Ignores execution time."""
    tracemalloc.start()
    try:
        func(*args, **kwargs)
    except Exception:
        pass # We just want to see how high memory got before it crashed/finished
        
    _, peak_mem = tracemalloc.get_traced_memory()
    tracemalloc.stop()
    return peak_mem / (1024 * 1024) # Return MB

def select_random_target(bn):
    # list all binary variables in the BN
    binary_vars = []
    for node in bn.nodes():
        cpd = bn.get_cpds(node)
        if cpd is not None and len(cpd.state_names[node]) == 2:
            binary_vars.append(node)
    if not binary_vars:
        return None, None
    print(f"Binary variables in {bn.name}: {binary_vars}")
    selected_target = random.choice(binary_vars)
    return selected_target

def run_targeted_sdp_experiment(output_csv="targeted_sdp_benchmark.csv"):
    
    results = []
    raw_results = []
    #H_RATIO = 0.20
    DECISION_THRESHOLD = 0.5
    TARGET_BUCKETS = [0.5, 0.6, 0.8, 1.0]
    MCMC_TRIALS = 10 
    
    for bn in models_to_run:
        n_nodes = bn.number_of_nodes()
        print(f"\n========================================")
        print(f"Processing BN: {bn.name}")
        
        all_nodes = list(bn.nodes())
        
        target = get_target(bn)
        #target = select_random_target(bn)
        if target is None:
            print(f"--> No binary target defined for {bn.name}, skipping.")
            continue
        target_states = bn.get_cpds(target).state_names[target]
        target_value = target_states[1] if len(target_states) > 1 else target_states[0]
        print(f"Target Node: {target}, Target Value: {target_value}")
        
        available_nodes = [n for n in all_nodes if n != target]
        print(f"H ratio: {get_h_ratio(bn)}")
        #n_hidden = max(1, int(len(available_nodes) * get_h_ratio(bn)))
        
        n_hidden = 10
        
        print(f"using {n_hidden} H variables")
        n_evidence = len(available_nodes) - n_hidden
        print(f"and {n_evidence} evidence variables")
        #hidden_vars = random.sample(available_nodes, n_hidden)
        #evidence_vars = [n for n in available_nodes if n not in hidden_vars]
        
            
        harvested_data = find_exact_experimental_patients_random(bn, target, target_value, DECISION_THRESHOLD,
                                                          n_evidence, buckets=TARGET_BUCKETS, batch_size=200)
        
        # Now process whatever it managed to find
        for target_sdp, result in harvested_data.items():
            if result is None:
                continue # We didn't find a patient for this specific bucket in this network
                
            patient, exact_sdp = result
            hidden_vars = [n for n in bn.nodes() if n not in patient and n != target]
            print(f"\n  -> Benchmarking found patient for bucket {target_sdp} (Exact: {exact_sdp:.4f})")
            
            # ========================================================
            # EXACT SDP EVALUATION
            # ========================================================
            partitions = get_partitions(bn, hidden_vars, target, patient)
            print(f"       -> Running Exact SDP...")
            
            # Pass 1: Time
            exact_sdp, exact_time, exact_success = run_for_time(
                fast_broadcast_sdp, bn, target, target_value, patient, DECISION_THRESHOLD, partitions
            )
            
            # Pass 2: Memory
            exact_mem_mb = run_for_memory(
                fast_broadcast_sdp, bn, target, target_value, patient, DECISION_THRESHOLD, partitions
            )
            
            if exact_success:
                print(f"          Time: {exact_time:.4f} sec | Peak Memory: {exact_mem_mb:.2f} MB")
            else:
                print(f"          [FAILED]: Crashed at {exact_mem_mb:.2f} MB")

            # ========================================================
            # MCMC EVALUATION
            # ========================================================
            mcmc_estimates = []
            mcmc_times = []
            
            print(f"       -> Running MCMC SDP (Trials: {MCMC_TRIALS})...")
            
            # Pass 1: Pure Time (across all trials)
            for trial in range(MCMC_TRIALS):
                est_sdp, t_time, _ = run_for_time(
                    fast_mcmc_sdp_estimation_new, bn, target, target_value, patient, DECISION_THRESHOLD,
                    n_samples=1000, burn_in=2000, thinning=50
                )
                mcmc_estimates.append(est_sdp)
                mcmc_times.append(t_time)
                
            mcmc_mean = np.mean(mcmc_estimates)
            mcmc_avg_time = np.mean(mcmc_times)
            mcmc_variance = np.var(mcmc_estimates)

            # Pass 2: Peak Memory
            mcmc_mem_mb = run_for_memory(
                fast_mcmc_sdp_estimation_new, bn, target, target_value, patient, DECISION_THRESHOLD,
                n_samples=100, burn_in=50, thinning=5
            )
            
            print(f"          Avg Time: {mcmc_avg_time:.4f} sec | Peak Memory: {mcmc_mem_mb:.2f} MB")
            
            absolute_error = abs(exact_sdp - mcmc_mean)

            # ========================================================
            # PARALLEL TEMPERING MCMC EVALUATION
            # ========================================================

            #pt_mcmc_estimates = []
            #pt_mcmc_times = []
#
            #print(f"       -> Running Parallel Tempering MCMC SDP (Trials: {MCMC_TRIALS})...")              
            #
            ## Pass 1: Pure Time (across all trials)
            #for trial in range(MCMC_TRIALS):
            #    est_sdp, t_time, _ = run_for_time(
            #        pt_mcmc_sdp_estimation, bn, target, target_value, patient, DECISION_THRESHOLD,
            #        n_samples=1000, burn_in=2000, thinning=50, n_chains=4, max_temp=40.0
            #    )
            #    pt_mcmc_estimates.append(est_sdp)
            #    pt_mcmc_times.append(t_time)
#
            #pt_mcmc_mean = np.mean(pt_mcmc_estimates)
            #pt_mcmc_avg_time = np.mean(pt_mcmc_times)
            #pt_mcmc_variance = np.var(pt_mcmc_estimates)
#
            ## Pass 2: Peak Memory
            #pt_mcmc_mem_mb = run_for_memory(
            #    pt_mcmc_sdp_estimation, bn, target, target_value, patient, DECISION_THRESHOLD,
            #    n_samples=100, burn_in=50, thinning=5, n_chains=4, max_temp=10.0
            #)
#
            #print(f"          Avg Time: {pt_mcmc_avg_time:.4f} sec | Peak Memory: {pt_mcmc_mem_mb:.2f} MB")
#
            #absolute_error_pt = abs(exact_sdp - pt_mcmc_mean)
            
        
            # Record everything to the dataset
            results.append({
                'Network': bn.name,
                'N_Nodes': n_nodes,
                'Target_Bucket': target_sdp,
                'Target_Node': target,
                'Target_Value': target_value,
                'Exact_SDP': exact_sdp,
                'Exact_Time_sec': exact_time,
                'MCMC_Mean_SDP': mcmc_mean,
                'MCMC_Variance': mcmc_variance,
                'MCMC_Avg_Time_sec': mcmc_avg_time,
                'Absolute_Error': absolute_error
                #'PT_MCMC_Mean_SDP': pt_mcmc_mean,
                #'PT_MCMC_Variance': pt_mcmc_variance,
                #'PT_MCMC_Avg_Time_sec': pt_mcmc_avg_time,
                #'PT_Absolute_Error': absolute_error_pt
            })
            
            # Save progressively
            pd.DataFrame(results).to_csv(output_csv, index=False)
            pd.DataFrame(raw_results).to_csv("raw_" + output_csv, index=False)

    print(f"\nExperiment Complete! Results saved to {output_csv}")
    return pd.DataFrame(results)

In [24]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning, module="pgmpy")
run_targeted_sdp_experiment()


Processing BN: child
Target Node: Sick, Target Value: no
H ratio: 0.5
using 10 H variables
and 9 evidence variables

Hunting for patients... (Locking 9 variables as evidence)
Generating batch 1/2 of 200 random realities...
--> Filled bucket 0.6 with Exact SDP: 0.5794
--> Filled bucket 1.0 with Exact SDP: 1.0000
--> Filled bucket 0.5 with Exact SDP: 0.5310
--> Filled bucket 0.8 with Exact SDP: 0.8105
All buckets filled successfully!

  -> Benchmarking found patient for bucket 0.5 (Exact: 0.5310)
       -> Running Exact SDP...
          Time: 0.0061 sec | Peak Memory: 0.06 MB
       -> Running MCMC SDP (Trials: 10)...
          Avg Time: 1.0512 sec | Peak Memory: 0.29 MB

  -> Benchmarking found patient for bucket 0.6 (Exact: 0.5794)
       -> Running Exact SDP...
          Time: 0.0127 sec | Peak Memory: 0.77 MB
       -> Running MCMC SDP (Trials: 10)...
          Avg Time: 1.1000 sec | Peak Memory: 0.29 MB

  -> Benchmarking found patient for bucket 0.8 (Exact: 0.8105)
       -> Runni

/home/joao/anaconda3/envs/bn-medical/lib/python3.8/site-packages/pgmpy/factors/discrete/DiscreteFactor.py:478: RuntimeWarning: invalid value encountered in divide
  phi.values = phi.values / phi.values.sum()


--> Filled bucket 0.6 with Exact SDP: 0.6427
--> Filled bucket 0.8 with Exact SDP: 0.7800
All buckets filled successfully!

  -> Benchmarking found patient for bucket 0.5 (Exact: 0.5419)
       -> Running Exact SDP...
          Time: 0.0104 sec | Peak Memory: 0.11 MB
       -> Running MCMC SDP (Trials: 10)...
          Avg Time: 1.1349 sec | Peak Memory: 0.38 MB

  -> Benchmarking found patient for bucket 0.6 (Exact: 0.6427)
       -> Running Exact SDP...
          Time: 0.0131 sec | Peak Memory: 0.48 MB
       -> Running MCMC SDP (Trials: 10)...
          Avg Time: 1.9435 sec | Peak Memory: 0.43 MB

  -> Benchmarking found patient for bucket 0.8 (Exact: 0.7800)
       -> Running Exact SDP...
          Time: 0.0101 sec | Peak Memory: 0.11 MB
       -> Running MCMC SDP (Trials: 10)...
          Avg Time: 1.3182 sec | Peak Memory: 0.38 MB

  -> Benchmarking found patient for bucket 1.0 (Exact: 1.0000)
       -> Running Exact SDP...
          Time: 0.0147 sec | Peak Memory: 0.11 MB
      

/home/joao/anaconda3/envs/bn-medical/lib/python3.8/site-packages/pgmpy/factors/discrete/DiscreteFactor.py:478: RuntimeWarning: invalid value encountered in divide
  phi.values = phi.values / phi.values.sum()


Generating batch 2/2 of 200 random realities...
Finished searching. Could not find patients for buckets: [0.5, 0.6, 0.8, 1.0]

Experiment Complete! Results saved to targeted_sdp_benchmark.csv


,Network,N_Nodes,Target_Bucket,Target_Node,Target_Value,Exact_SDP,Exact_Time_sec,MCMC_Mean_SDP,MCMC_Variance,MCMC_Avg_Time_sec,Absolute_Error
0,child,20,0.5,Sick,no,0.531043,0.006088,0.5347,0.000273,1.051176,0.003657
1,child,20,0.6,Sick,no,0.579392,0.012697,0.5793,0.000791,1.100034,0.000092
2,child,20,0.8,Sick,no,0.810535,0.012032,0.8005,0.000696,1.151947,0.010035
3,child,20,1.0,Sick,no,1.000000,0.007608,1.0000,0.000000,1.118255,0.000000
4,alarm,37,0.5,HYPOVOLEMIA,FALSE,0.541886,0.010442,0.5401,0.000233,1.134900,0.001786
5,alarm,37,0.6,HYPOVOLEMIA,FALSE,0.642670,0.013098,0.6456,0.000256,1.943502,0.002930
6,alarm,37,0.8,HYPOVOLEMIA,FALSE,0.779957,0.010068,0.7884,0.000227,1.318211,0.008443
7,alarm,37,1.0,HYPOVOLEMIA,FALSE,1.000000,0.014741,1.0000,0.000000,1.528129,0.000000


In [25]:
# generate random patient for hailfinder and run MCMC on it
all_nodes = list(hailfinder_model.nodes())
target = get_target(hailfinder_model)
target_states = hailfinder_model.get_cpds(target).state_names[target]
target_value = target_states[1] if len(target_states) > 1 else target_states[0]
available_nodes = [n for n in all_nodes if n != target]
n_hidden = 49
n_evidence = len(available_nodes) - n_hidden

# Randomly pick evidence variables and assign them random states
evidence_vars = random.sample(available_nodes, n_evidence)
patient = {
    var: random.choice(hailfinder_model.get_cpds(var).state_names[var])
    for var in evidence_vars
}

print(f"Target: {target} = {target_value}")
print(f"Evidence ({n_evidence} vars), hidden ({n_hidden} vars)")

# Run MCMC and time it
start = time.time()
mcmc_estimate = fast_mcmc_sdp_estimation_new(
    hailfinder_model, target, target_value, patient, threshold=0.5,
    n_samples=1000, burn_in=1000, thinning=50
)
elapsed = time.time() - start
print(f"MCMC SDP estimate: {mcmc_estimate:.4f} | Time: {elapsed:.2f} sec")


Target: ScenRelAMCIN = CThruK
Evidence (6 vars), hidden (49 vars)
MCMC SDP estimate: 1.0000 | Time: 4.32 sec
